# Automotive: CAN, ISO-TP, UDS and OBD-II

A scan tool and a simulated engine ECU share a virtual CAN bus. Swap the URI for `socketcan:can0` or `slcan:COM5` to talk to real hardware — only on vehicles you are authorised to service.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.13.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.13.0-preview.1"
#r "nuget: IoTCom.Net.Protocols.Uds, 0.13.0-preview.1"

## CAN frames in candump notation
Frames are validated (11/29-bit identifiers, classic or CAN FD lengths).

In [ ]:
using IoTCom.Net.Transport.Can;

var request = CanFrame.Parse("7DF#02010C");   // OBD-II: engine rpm
Console.WriteLine($"{request} → id 0x{request.Id:X3}, {request.Data.Length} bytes, DLC {request.Dlc}");
var fd = new CanFrame(0x18DAF110, CanDlc.Pad(new byte[20]), CanFrameFlags.Extended | CanFrameFlags.Fd);
Console.WriteLine($"CAN FD: {fd.Data.Length} bytes (padded), DLC {fd.Dlc}");

## An ECU simulator and a scan tool on a virtual bus

In [ ]:
using IoTCom.Net.Protocols.Uds;

var net = new VirtualCanNetwork();
await using var ecu = EcuSimulator.Create(net.CreateNode());
await ecu.StartAsync();
var bus = net.CreateNode();
await bus.ConnectAsync();
using var sniffer = bus.OpenReader();

await using var obd = ObdClient.Create(bus);
await obd.ConnectAsync();
foreach (var pid in new[] { ObdPids.EngineRpm, ObdPids.CoolantTemperature, ObdPids.ModuleVoltage })
    Console.WriteLine(await obd.ReadPidAsync(pid));
Console.WriteLine($"VIN {await obd.ReadVinAsync()}");

## ISO-TP on the wire
The 17-character VIN needs a first frame, a flow control and consecutive frames — segmented by the Rust state machine.

In [ ]:
while (sniffer.TryRead(out var f)) if (f.Data.Length > 0 && f.Data.Span[0] >> 4 is 1 or 2 or 3) Console.WriteLine(f);

## UDS: identification, trouble codes and security access

In [ ]:
await using var uds = UdsClient.Create(bus);
await uds.ConnectAsync();
Console.WriteLine(await uds.ReadStringAsync(UdsDid.SoftwareVersion));
foreach (var dtc in await uds.ReadDtcsAsync()) Console.WriteLine($"{dtc}  {dtc.Status}");

try { await uds.WriteDataByIdentifierAsync(UdsDid.RepairShopCode, "WS-01"u8.ToArray()); }
catch (UdsNegativeResponseException ex) { Console.WriteLine(ex.Message); }

await uds.StartSessionAsync(UdsSession.Extended);
await uds.SecurityAccessAsync(0x01, EcuSimulator.ComputeKey);
await uds.WriteDataByIdentifierAsync(UdsDid.RepairShopCode, "WS-01"u8.ToArray());
Console.WriteLine(await uds.ReadStringAsync(UdsDid.RepairShopCode));

## Going further
The Gallery demo *Vehicle diagnostics* shows the same stack with a live dashboard; the CLI has `iotcom obd live --can sim`. See `docs/en/protocols/uds.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*